In [1]:
import pandas as pd
import numpy as np
import re, pickle, gzip

In [2]:
cutoffs = pd.read_excel('noncanonical_groupwalk_qvalue_thresholds.xlsx')
cutoffs = cutoffs[cutoffs.group_type=='PTM']
# cutoffs.q_sel_final = np.round(cutoffs.q_sel_final, 3)
cutoffs.q_sel_final = cutoffs.apply(lambda x: 0.1 if x["fail_sel_cutoff"] else np.round(x["fdp_cutoff_q_max"]/100,3), axis=1)
cutoffs.reset_index(drop=True, inplace=True)
cutoffs.head()

,sample,subset,dataset,database,search_type,group_type,approach,min_fdp,max_score_FDP_cutoff,fdp_cutoff,fdp_cutoff_q_min,fdp_cutoff_q_max,fail_sel_cutoff,q_sel_final,fdp_final
0,noncanonical_130327_o2_01_hu_C1_2hr_fdp,noncanonical,PXD002057,open,ClosedSearch,PTM,groupwalk,22.222222,NaN,NaN,NaN,NaN,True,0.10,22.222222
1,noncanonical_130327_o2_02_hu_P1_2hr_fdp,noncanonical,PXD002057,open,ClosedSearch,PTM,groupwalk,14.574899,NaN,NaN,NaN,NaN,True,0.10,14.574899
2,noncanonical_130327_o2_03_hu_C2_2hr_fdp,noncanonical,PXD002057,open,ClosedSearch,PTM,groupwalk,0.000000,1.375027,0.757576,2.090909,3.034352,False,0.03,0.757576
3,noncanonical_130327_o2_04_hu_P2_2hr_fdp,noncanonical,PXD002057,open,ClosedSearch,PTM,groupwalk,0.000000,1.089044,0.000000,8.342986,11.947101,True,0.10,0.000000
4,noncanonical_130327_o2_05_hu_C3_2hr_fdp,noncanonical,PXD002057,open,ClosedSearch,PTM,groupwalk,0.000000,2.028668,0.724638,2.952756,2.955388,False,0.03,0.724638


In [3]:
cutoffs.database = 'openprot'
cutoffs.subset   = 'NonCanonical'

In [4]:
def convert_sample_names(x):
    x = re.sub(r'^noncanonical_', '', x)
    x = re.sub(r'_fdp$', '.mgf', x)
    return x

cutoffs['sample'] = cutoffs['sample'].apply(convert_sample_names)

In [5]:
for _ in ['sample','dataset','subset','database','search_type','group_type','approach']:
    print(_, len(set(cutoffs[_])), set(cutoffs[_]))

sample 24 {'130327_o2_06_hu_P3_2hr.mgf', '130327_o2_05_hu_C3_2hr.mgf', '130327_o2_04_hu_P2_2hr.mgf', 'AM8.mgf', 'Sample-BT474.mgf', 'AM19.mgf', 'AM18.mgf', 'AM7.mgf', 'Sample-MCF.mgf', 'AM10.mgf', 'SampleHela.mgf', 'AM9.mgf', '130327_o2_02_hu_P1_2hr.mgf', 'AM12.mgf', 'AM11.mgf', 'AM16.mgf', '130327_o2_03_hu_C2_2hr.mgf', 'AM13.mgf', 'AM21.mgf', 'AM17.mgf', 'AM15.mgf', 'AM14.mgf', 'AM20.mgf', '130327_o2_01_hu_C1_2hr.mgf'}
dataset 3 {'PXD014258', 'PXD005833', 'PXD002057'}
subset 1 {'NonCanonical'}
database 1 {'openprot'}
search_type 2 {'ClosedSearch', 'OpenSearch'}
group_type 1 {'PTM'}
approach 1 {'groupwalk'}


In [6]:
cutoffs = cutoffs[['q_sel_final','sample','dataset','database','search_type','subset']].copy()
cutoffs2 = cutoffs.set_index(['sample','subset','search_type']).to_dict()['q_sel_final']

In [7]:
with gzip.open('custom-groupwalk-cutoffs.gz','wb') as outfile:
    pickle.dump(cutoffs2, outfile)

In [8]:
with gzip.open('custom-groupwalk-cutoffs.gz','rb') as infile:
    b = pickle.load(infile)

In [9]:
cutoffs2 == b

True